# Validation 2 — Kiểm định hoán vị cho hệ số nhân quả đuôi (20 mã)

| Kiểm định | $H_0$ | Cách sinh dữ liệu dưới $H_0$ | Thống kê |
|---|---|---|---|
| (a) Độc lập | các mã độc lập ở đuôi | xáo trộn thời gian **từng cột riêng** | $\hat\Psi_{jk}$, một phía |
| (b) Không có chiều | phân phối đồng thời của $(F_j(X_j), F_k(X_k))$ đối xứng | hoán đổi hai biến trong mỗi phiên với xác suất 1/2 (thang hạng) | $D_{jk} = \hat\Psi_{jk}-\hat\Psi_{kj}$, hai phía |

$p = (1 + \#\{T^{(b)} \text{ cực đoan bằng/hơn}\})/(B+1)$. Hiệu chỉnh nhiều kiểm định: (a) **max-T** của Westfall & Young (1993) — so mỗi $\hat\Psi_{jk}$
với phân phối của $\max$ trên cả 380 cặp dưới cùng hoán vị (kiểm soát FWER; Holm cần $p < 0{,}05/380 \approx 1{,}3\cdot10^{-4}$, không đạt được với $B = 500$);
(b) mỗi cặp có hoán vị riêng nên dùng **Benjamini–Hochberg** (FDR) và Holm, với $B = 4.999$.
Kiểm định trên $z$ và trên phần dư $e$ (đã loại thị trường leave-one-out). Các kiểm định chạy song song (joblib), mỗi cặp có seed riêng.

## 0. Thiết lập

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import eda_utils as U
import causal_tail as CT
import causal_baselines as CB
import tail_preprocessing as TP

U.apply_publication_style()
pd.set_option("display.float_format", "{:,.3f}".format)
SYMBOLS, COL, PAL = U.SYMBOLS, U.SYMBOL_COLORS, U.PALETTE
FOLDER = "validation"                      # hình → report/figure/processing, bảng → report/table/processing
METHOD_COLORS = {"EASE": PAL["blue_main"], "Pairwise LiNGAM": PAL["red_strong"], "ICA-LiNGAM": PAL["green_3"],
                 "Rank PC": PAL["violet"], "Ngẫu nhiên": PAL["neutral"]}

import validation as V
from scipy import stats
P = len(SYMBOLS)
OFF = ~np.eye(P, dtype=bool)

Z = pd.read_parquet(ROOT / "data" / "processed" / "garch_std_resid_market.parquet")
z = Z[SYMBOLS]
n, K = len(z), CT.default_k(len(z))
E, beta = V.loo_market_adjust(z)                 # phần dư sau khi loại thị trường leave-one-out
print(f"n = {n} phiên ({Z.index[0]:%d/%m/%Y} → {Z.index[-1]:%d/%m/%Y}), {P} mã, k = {K}")

B_IND, B_SWAP = 500, 4999

## 1. Kiểm định độc lập

**Các bước:** với $z$ và $e$: 500 bộ dữ liệu xáo trộn → ma trận $\hat\Psi^{(b)}$ (cài đặt vector hoá, ~20 ms/ma trận); p-value một phía
hiệu chỉnh max-T cho 380 cặp. Hình: ma trận các cặp bác bỏ $H_0$ (max-T 5%) theo ngành.

In [ ]:
res = {}
for nm, X in {"z": z, "e": E}.items():
    obs = CT.tail_coef_matrix(X, k=K).to_numpy()
    null = V.permutation_null_independence(X, k=K, B=B_IND, seed=3)
    ph = V.maxT_pvalues(obs, null)
    res[nm] = (obs, ph)
    print(f"{nm}: ngưỡng max-T 5% của Ψ̂ = {np.quantile(np.nanmax(null[:, OFF], axis=1), 0.95):.3f}")
ind = pd.DataFrame({nm: {"psi_TB": o[OFF].mean(), "so_cap_bac_bo_maxT5%": int((h[OFF] < 0.05).sum())}
                    for nm, (o, h) in res.items()}).T
U.save_table(ind, "02_independence_summary", FOLDER)
for nm, (o, h) in res.items():
    U.save_table(pd.DataFrame(h, index=SYMBOLS, columns=SYMBOLS), f"02_independence_pholm_{nm}", FOLDER)

fig, axes = plt.subplots(1, 2, figsize=(17, 7.5))
for ax, (nm, (o, h)) in zip(axes, res.items()):
    sig = pd.DataFrame(np.where(OFF, (h < 0.05).astype(float), np.nan), index=SYMBOLS, columns=SYMBOLS)
    U.make_heatmap(ax, sig, cmap="Blues", vmin=0, vmax=1, fmt=None, tick_fontsize=9, cbar_label="Bác bỏ H0 (max-T 5%)",
                   title=f"{nm}: {int(np.nansum(sig.values))}/380 cặp phụ thuộc đuôi")
    ax.set_xlabel("Hệ quả"); ax.set_ylabel("Nguyên nhân")
U.finalize_figure(fig, "02_independence_test", folder=FOLDER); plt.show()
ind

### Phân tích

_(điền sau khi chạy)_

## 2. Kiểm định chiều (hoán đổi đối xứng) cho 190 cặp

**Các bước:** cho 190 cặp trên $z$, 190 cặp trên $e$, và 20 cặp (VNI, mã) trên $z$: $D$ quan sát và $B = 4.999$ giá trị dưới $H_0$; p hai phía;
q-value Benjamini–Hochberg và Holm trong từng họ.
Hình: phân phối p-value (dưới $H_0$ đúng ở mọi cặp, histogram phẳng) và các cặp có p nhỏ nhất.

In [ ]:
pairs = [(a, b) for i, a in enumerate(SYMBOLS) for b in SYMBOLS[i + 1:]]
sw = pd.concat([V.swap_test_pairs(z, pairs, k=K, B=B_SWAP, seed=100).assign(ho="z"),
                V.swap_test_pairs(E, pairs, k=K, B=B_SWAP, seed=5000).assign(ho="e"),
                V.swap_test_pairs(Z, [("VNI", s) for s in SYMBOLS], k=K, B=B_SWAP, seed=9000).assign(ho="VNI–mã")],
               ignore_index=True)
U.save_table(sw.set_index(["ho", "cap"]), "02_swap_test", FOLDER)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
for ax, fam in zip(axes, ["z", "e", "VNI–mã"]):
    d = sw[sw.ho == fam]
    ax.hist(d.p, bins=np.linspace(0, 1, 21), color=PAL["blue_secondary"], edgecolor="white")
    ax.axhline(len(d) / 20, color=PAL["red_strong"], ls="--", lw=1, label="kỳ vọng dưới H0")
    ax.set_title(f"{fam}: {int((d.p < 0.05).sum())}/{len(d)} có p < 0,05; {int((d.q_bh < 0.1).sum())} có q < 0,1", fontsize=11)
    ax.set_xlabel("p-value")
axes[0].set_ylabel("Số cặp"); axes[0].legend(fontsize=9)
U.finalize_figure(fig, "02_swap_test", folder=FOLDER); plt.show()
sw.groupby("ho").apply(lambda g: pd.Series({"so_cap": len(g), "p<0,05": int((g.p < 0.05).sum()),
                                           "q_BH<0,1": int((g.q_bh < 0.1).sum()), "Holm<0,05": int((g.p_holm < 0.05).sum())}))

In [ ]:
sw.sort_values("p").groupby("ho").head(6)[["ho", "cap", "D", "p", "q_bh", "p_holm", "chieu"]].round(4)

### Phân tích

_(điền sau khi chạy)_

## 3. Kết luận

_(điền sau khi chạy)_